In [0]:
# Databricks notebook source

class Config:
    def __init__(self):

        # Noms Unity Catalog des External Locations
        self.data_location = "data_zone"
        self.checkpoint_location = "checkpoint_zone"

        # On récupère automatiquement les vraies URLs ADLS.
        # Ainsi aucun chemin ADLS n'est codé en dur.
        self.base_dir_data = (
            spark.sql(
                f"DESCRIBE EXTERNAL LOCATION `{self.data_location}`"
            )
            .select("url")
            .collect()[0][0]
            .rstrip("/")
        )

        self.base_dir_checkpoint = (
            spark.sql(
                f"DESCRIBE EXTERNAL LOCATION `{self.checkpoint_location}`"
            )
            .select("url")
            .collect()[0][0]
            .rstrip("/")
        )

        # Architecture Medallion
        self.bronze_schema = "bronze"
        self.silver_schema = "silver"
        self.gold_schema = "gold"

        # Autres paramètres utilisés plus tard
        self.maxFilesPerTrigger = 1000

### Description de 01-config

Ce notebook centralise les paramètres de configuration du projet dans une classe `Config`.

Il récupère automatiquement les chemins physiques des External Locations `data_zone` et `checkpoint_zone` depuis Unity Catalog grâce à la commande :

`DESCRIBE EXTERNAL LOCATION`

Cela évite de coder en dur les URLs ADLS Gen2 dans les notebooks. Si l’emplacement physique change, il suffit de modifier l’External Location dans Unity Catalog sans changer le code.

- `base_dir_data` contient le chemin de la zone de données utilisée pour les fichiers entrants.
- `base_dir_checkpoint` contient le chemin utilisé pour stocker les checkpoints des traitements streaming.
- `.select("url").collect()[0][0]` récupère uniquement l’URL retournée par Databricks.
- `.rstrip("/")` supprime le `/` final afin de faciliter la construction de sous-chemins.

Ce notebook sert donc de **configuration centrale et réutilisable** pour les autres composants du projet, notamment le setup, l’ingestion Bronze et les traitements streaming.